# 🎯 캡스톤 — 이 에이전트, 도입해도 되나

11부 졸업 시험입니다. **새 개념은 없습니다**. 워밍업~11장의 도구를 총조립해
가상 제품 '일손 2' 도입 제안서의 네 주장을 검산대 4곳에서 감정합니다.
웹 챕터(CapstoneAgentLab)와 **같은 수치**를 파이썬으로 직접 계산해 확인해요.

> 🤖 **AI 튜터와 함께**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다
> (로컬 Ollama 가 꺼져 있어도 노트북 학습엔 지장 없음).

In [ ]:
from tutor import chat
chat('ch12')  # 이 챕터를 아는 AI 튜터

## 검산대 ① “각 단계 98%, 어떤 업무든”: 사슬 성공률로 환산 (워밍업·2장)

걸음 정확도 98%를 업무의 **단계 수**만큼 곱하면 실제 성공률이 나옵니다: $P = 0.98^{n}$.

In [ ]:
for n in [4, 8, 12, 20]:
    print(f'{n:>2}단계: 0.98^{n} = {0.98**n*100:.1f}%')

assert round(0.98**4 * 100, 1) == 92.2
assert round(0.98**8 * 100, 1) == 85.1
assert round(0.98**12 * 100, 1) == 78.5   # 사내 대표 업무 평균 12단계
assert round(0.98**20 * 100, 1) == 66.8
print('\n→ 대표 업무 12단계면 78.5%. 5건 중 1건 이상 실패. “어떤 업무든”은 단계 수를 감춘 과장.')

## 검산대 ② “20건 90% 데모”: 오차범위 안인가 (11장)

데모는 표본입니다. 냅킨 공식(최악값 기준) $\pm \approx 98/\sqrt{n}$ 로 오차범위를 봅니다.

In [ ]:
import math

for n in [20, 50, 200, 1000]:
    m = 98 / math.sqrt(n)
    lo = 90 - m
    hi = min(90 + m, 100)
    over = '+' if 90 + m > 100 else ''
    tag = ' ← 70%짜리와 겹침!' if lo < 75 else ''
    print(f'{n:>4}건: ±{m:.1f}%p → [{lo:.1f}, {hi:.1f}{over}]{tag}')

assert round(98 / math.sqrt(20), 1) == 21.9
assert round(90 - 98 / math.sqrt(20), 1) == 68.1   # 하한 — 70%와 구별 불가
assert round(98 / math.sqrt(200), 2) == 6.93       # 도입 판단엔 최소 200건
print('\n→ 20건의 90%는 ±21.9%p라 70%짜리와 구별 불가. 200건(±6.9%p) 요구 → 보류.')

## 검산대 ③ “pass@5 = 97%”: 한 번에 맞을 확률은? (4·11장)

pass@5 는 5번 시도해 한 번이라도 성공: $1-(1-p)^5$. 이를 **pass@1**($p$)로 역산합니다:
$p = 1 - (1-\text{pass@5})^{1/5}$.

In [ ]:
def pass1_from_pass5(p5):
    return 1 - (1 - p5) ** (1 / 5)

for p5 in [0.90, 0.97, 0.99]:
    print(f'pass@5 = {p5*100:.0f}%  →  pass@1 = {pass1_from_pass5(p5)*100:.1f}%')

assert round(pass1_from_pass5(0.97) * 100, 1) == 50.4   # 한 번에 맞을 확률은 절반뿐
# 역방향 검증: pass@1 50.4%를 5번 굴리면 다시 97%
p1 = pass1_from_pass5(0.97)
assert round((1 - (1 - p1) ** 5) * 100) == 97
print('\n→ 광고의 97%는 5번 굴린 결과. 한 번에 맞을 확률은 50.4%.')
print('  활용하려면 성공을 골라낼 판별 수단 + 비용 5배(7장)가 전제 → 보류.')

## 검산대 ④ “자체 검수로 이중 확인”: 얼마나 믿을 수 있나 (6·9장)

“통과” 도장의 신뢰도 = 베이즈 사후확률. 검수자 민감도 $s=0.9$·특이도 $t=0.8$ 일 때
$P(\text{정답}\mid\text{통과}) = \dfrac{p\,s}{p\,s + (1-p)(1-t)}$. 업무 난이도(기저율 $p$)에 좌우됩니다.

In [ ]:
def posterior(p, s=0.9, t=0.8):
    return p * s / (p * s + (1 - p) * (1 - t))

for label, p in [('쉬운 업무', 0.80), ('보통 업무', 0.35), ('어려운 업무', 0.05)]:
    print(f'{label} (p={p*100:.0f}%): 통과 시 정답 {posterior(p)*100:.1f}%')

assert round(posterior(0.80) * 100, 1) == 94.7
assert round(posterior(0.35) * 100, 1) == 70.8
assert round(posterior(0.05) * 100, 1) == 19.1   # 기저율의 역설
print('\n→ 어려운 업무는 검수 통과해도 19.1%뿐(기저율의 역설). 검수자 독립성도 미확인 → 조건부.')

## 감정 결과를 그림으로: 단계 수 곡선과 pass@k 역산

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

# ① 걸음 98%의 단계 수별 업무 성공률
ns = np.arange(1, 25)
ax1.plot(ns, 0.98 ** ns * 100, marker='o', ms=3, color='#6366f1')
ax1.axvline(12, ls=':', color='#94a3b8')
ax1.annotate('12단계\n78.5%', (12, 78.5), textcoords='offset points', xytext=(8, 6), color='#e11d48')
ax1.set_title('검산대 ① 걸음 98% → 업무 성공률 0.98^n')
ax1.set_xlabel('업무 단계 수 n'); ax1.set_ylabel('업무 성공률 (%)')
ax1.set_ylim(60, 100); ax1.grid(alpha=0.3)

# ③ pass@5 → pass@1 역산
p5s = np.linspace(0.80, 0.999, 100)
p1s = 1 - (1 - p5s) ** (1 / 5)
ax2.plot(p5s * 100, p1s * 100, color='#f59e0b')
ax2.plot([97], [50.4], marker='o', ms=7, color='#e11d48')
ax2.annotate('pass@5 97%\n→ pass@1 50.4%', (97, 50.4), textcoords='offset points', xytext=(-96, 6), color='#e11d48')
ax2.plot([80, 100], [80, 100], ls='--', lw=1, color='#cbd5e1')  # y=x 기준선
ax2.set_title('검산대 ③ pass@5 → pass@1 역산 (항상 아래)')
ax2.set_xlabel('pass@5 (%)'); ax2.set_ylabel('pass@1 = 한 번에 맞을 확률 (%)')
ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 최종 판정서: ‘일손 2’ 도입 제안 감정 결과

| 주장 | 감정 | 판정 |
|---|---|---|
| ① 각 단계 98% (“어떤 업무든”) | 12단계면 78.5% | ⚠ 단계 수 감춘 과장 |
| ② 20건 90% 데모 | ±21.9%p → [68.1, 100+] | ⚠ 표본 부족: 보류 |
| ③ pass@5 97% | pass@1 50.4% + 판별·비용 전제 | ⚠ 전제 미충족: 보류 |
| ④ 자체 검수 이중 확인 | 난이도별 19~95% + 독립성 미확인 | ⚠ 조건부 |

**도입 판단:** “어떤 업무든 믿고 맡기세요”는 현재 자료로 성립하지 않음. 수치 조작은
없었지만 단계 수·표본·pass@k·검수 전제를 감춘 **과장**. **조건부 파일럿 제안**: 성공을
판별할 수 있는(테스트 가능한) 업무부터 **200건 시범** + **독립 검수 설계** + **단계별
성공률 로깅**(2장 병목 계측).

핵심 통찰: 도입 판단의 질문은 “숫자가 맞나”가 아니라 **“숫자가 그 결론을 지지하나”**입니다.

🎓 이 판정서를 스스로 쓸 수 있다면 **11부 졸업**입니다. 시리즈의 질문 사슬이 완성됐어요:
무엇을(6부) → 어떻게(8부) → 무엇으로(9부) → 얼마나·정말로(10부) → **이제 일을 맡겨도 되나(11부)**.

**웹 챕터로 돌아가 졸업 시험 10문제를 풀면 11부가 완결됩니다.**